In [7]:
import pymysql
from configparser import ConfigParser
from create_table import create_superstore_tables

# 讀取 .env 檔案取得資料庫連線資訊
config = ConfigParser()
config.read('../Chapter1/config.ini')

# 建立資料庫連線
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
)

with connection.cursor() as cursor:
    # 建立資料庫
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS superstore")

    cursor.execute(f"SHOW DATABASES")
    dbs = cursor.fetchall()
    print(dbs)

    # 建立資料表
    create_superstore_tables("superstore")

[{'Database': 'chapter2'}, {'Database': 'chapter3'}, {'Database': 'classicmodels'}, {'Database': 'information_schema'}, {'Database': 'my_database'}, {'Database': 'my_titanic'}, {'Database': 'my_train_titanic'}, {'Database': 'mysql'}, {'Database': 'performance_schema'}, {'Database': 'sakila'}, {'Database': 'social_media_app'}, {'Database': 'superstore'}, {'Database': 'sys'}, {'Database': 'testdb'}, {'Database': 'transaction_test'}, {'Database': 'world'}]
{'Tables_in_superstore': 'customers'}
{'Tables_in_superstore': 'orderdetails'}
{'Tables_in_superstore': 'orders'}
{'Tables_in_superstore': 'products'}


In [11]:
# 建立資料庫連線
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database='superstore'
)

In [4]:
import pandas as pd

# 讀取 csv 檔案
df = pd.read_csv("Sample-Superstore.csv", encoding="latin-1")
# print(df.columns)

# 提取客戶欄位的資料
customer = df[["Customer ID", "Customer Name", "Segment"]]

# 去除重複的客戶資料
customer = customer.drop_duplicates()

customer

,Customer ID,Customer Name,Segment
0,CG-12520,Claire Gute,Consumer
2,DV-13045,Darrin Van Huff,Corporate
3,SO-20335,Sean O'Donnell,Consumer
5,BH-11710,Brosina Hoffman,Consumer
12,AA-10480,Andrew Allen,Consumer
...,...,...,...
8666,CJ-11875,Carl Jackson,Corporate
9209,RS-19870,Roy Skaria,Home Office
9399,SC-20845,Sung Chung,Consumer
9441,RE-19405,Ricardo Emerson,Consumer


Customers

In [12]:
# 取得 df 中的客戶資料
customer

# 轉成 list 以便後續寫入
customer_list = customer.values.tolist()
print(customer_list)

# 寫入 Customers 資料表
with connection.cursor() as cursor:
    sql = """
        INSERT INTO customers (customer_id, customer_name, segment)
        VALUES (%s, %s, %s)
    """
    cursor.executemany(sql, customer_list)
    
    # 檢查寫入數量
    print(cursor.rowcount)

[['CG-12520', 'Claire Gute', 'Consumer'], ['DV-13045', 'Darrin Van Huff', 'Corporate'], ['SO-20335', "Sean O'Donnell", 'Consumer'], ['BH-11710', 'Brosina Hoffman', 'Consumer'], ['AA-10480', 'Andrew Allen', 'Consumer'], ['IM-15070', 'Irene Maddox', 'Consumer'], ['HP-14815', 'Harold Pawlan', 'Home Office'], ['PK-19075', 'Pete Kriz', 'Consumer'], ['AG-10270', 'Alejandro Grove', 'Consumer'], ['ZD-21925', 'Zuschuss Donatelli', 'Consumer'], ['KB-16585', 'Ken Black', 'Corporate'], ['SF-20065', 'Sandra Flanagan', 'Consumer'], ['EB-13870', 'Emily Burns', 'Consumer'], ['EH-13945', 'Eric Hoffmann', 'Consumer'], ['TB-21520', 'Tracy Blumstein', 'Consumer'], ['MA-17560', 'Matt Abelman', 'Home Office'], ['GH-14485', 'Gene Hale', 'Corporate'], ['SN-20710', 'Steve Nguyen', 'Home Office'], ['LC-16930', 'Linda Cazamias', 'Corporate'], ['RA-19885', 'Ruben Ausman', 'Corporate'], ['ES-14080', 'Erin Smith', 'Corporate'], ['ON-18715', 'Odella Nelson', 'Corporate'], ['PO-18865', "Patrick O'Donnell", 'Consumer'

In [13]:
connection.commit()

Orders 

In [28]:
from datetime import datetime

# 取得 df 中的訂單資料
# print(df.columns)
orders = df[['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID']].drop_duplicates()

# 將日期格式轉換為 datetime.date
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format='%m/%d/%Y').dt.date
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], format='%m/%d/%Y').dt.date

# 轉成 list 以便後續寫入
orders_list = orders.values.tolist()
print(orders_list)

# 寫入 Orders 資料表
with connection.cursor() as cursor:
    sql = """
        INSERT INTO orders (order_id, order_date, ship_date, ship_mode, customer_id)
        VALUES (%s, %s, %s, %s, %s)
    """
    cursor.executemany(sql, orders_list)

    print(cursor.rowcount)

[['CA-2016-152156', datetime.date(2016, 11, 8), datetime.date(2016, 11, 11), 'Second Class', 'CG-12520'], ['CA-2016-138688', datetime.date(2016, 6, 12), datetime.date(2016, 6, 16), 'Second Class', 'DV-13045'], ['US-2015-108966', datetime.date(2015, 10, 11), datetime.date(2015, 10, 18), 'Standard Class', 'SO-20335'], ['CA-2014-115812', datetime.date(2014, 6, 9), datetime.date(2014, 6, 14), 'Standard Class', 'BH-11710'], ['CA-2017-114412', datetime.date(2017, 4, 15), datetime.date(2017, 4, 20), 'Standard Class', 'AA-10480'], ['CA-2016-161389', datetime.date(2016, 12, 5), datetime.date(2016, 12, 10), 'Standard Class', 'IM-15070'], ['US-2015-118983', datetime.date(2015, 11, 22), datetime.date(2015, 11, 26), 'Standard Class', 'HP-14815'], ['CA-2014-105893', datetime.date(2014, 11, 11), datetime.date(2014, 11, 18), 'Standard Class', 'PK-19075'], ['CA-2014-167164', datetime.date(2014, 5, 13), datetime.date(2014, 5, 15), 'Second Class', 'AG-10270'], ['CA-2014-143336', datetime.date(2014, 8, 27

In [30]:
connection.commit()

Products

In [32]:
# 取得 df 中的產品資料
# print(df.columns)
products = df[['Product ID', 'Category', 'Sub-Category', 'Product Name']].drop_duplicates()

# 轉成 list 以便後續寫入
product_list = products.values.tolist()

# 寫入 Products 資料表
with connection.cursor() as cursor:
    sql = """
        INSERT INTO products (product_id, category, sub_category, product_name)
        VALUES (%s, %s, %s, %s)
    """
    cursor.executemany(sql, product_list)

    print(cursor.rowcount)

1894


In [41]:
connection.commit()

OrderDetails

In [37]:
# 取得 df 中的訂單明細資料
print(df.columns)
orderdetails = df[['Row ID', 'Order ID', 'Product ID', 'Sales', 'Quantity', 'Discount', 'Profit']].drop_duplicates()

# 轉成 list 以便後續寫入
orderdetails_list = orderdetails.values.tolist()

# 寫入 OrderDetails 資料表
with connection.cursor() as cursor:
    sql = """
        INSERT INTO orderdetails (row_id, order_id, product_id, sales, quantity, discount, profit)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """
    cursor.executemany(sql, orderdetails_list)

    print(cursor.rowcount)

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')
9994


In [40]:
connection.commit()